#  Deep Learning Data Audit and Benchmark Design

## Purpose

This notebook prepares the MarketGuard India dataset for deep learning experiments.

The goal is not to train a neural network immediately. The first goal is to establish a fair and leakage-safe experimental design so that deep learning models can be compared directly against the existing Random Forest baseline.

The current Random Forest models remain the production baseline until another model demonstrates a meaningful and stable improvement on untouched historical data.

---

## Current Production Baseline

MarketGuard currently uses two Random Forest classification pipelines:

### Outperformance Model

Predicts whether a stock will outperform the NIFTY 50 over the following 20 trading days.

Target:

`target_outperform_nifty50_20d`

### Downside Model

Predicts whether a stock will experience a downside event of at least 10% during the following 20 trading days.

Target:

`target_big_downside_10pct_20d`

Both production models use the same ordered set of 79 engineered features.

---

## Main Research Question

Can a deep learning model improve the out-of-time performance of the current MarketGuard models?

The comparison will include:

- Random Forest baseline
- Tabular multilayer perceptron
- Residual multilayer perceptron
- A tabular Transformer candidate
- Temporal models in a later phase
- Random Forest and neural-network ensembles

Deep learning will only be promoted when it improves the actual historical ranking and risk-screening behavior, not merely training accuracy.

---

## Primary Objectives

This notebook will:

1. Load and validate the research dataset.
2. Recover the exact ordered model-feature list.
3. Identify the original model-training configuration.
4. Inspect the available date range.
5. Inspect target availability and class balance.
6. Define chronological train, validation, and test periods.
7. Add a time gap around split boundaries where required.
8. Confirm that future and target columns are excluded from model inputs.
9. Record the current Random Forest benchmark.
10. Define the evaluation criteria for deep learning candidates.

---

## Experimental Rules

### Chronological Splits Only

Rows must not be randomly divided into train, validation, and test sets.

Financial observations close together in time are strongly related. Random splitting could place neighboring observations from the same stock into different datasets and create an unrealistically easy evaluation.

The experiments will use strict chronological periods:

```text
Training period
        ↓
Embargo or gap
        ↓
Validation period
        ↓
Embargo or gap
        ↓
Final test period
```

### Untouched Final Test Period

The final test period must not influence:

- Architecture selection
- Hyperparameter tuning
- Early stopping
- Feature selection
- Probability thresholds
- Ensemble weights

It will only be used after the model design is finalized.

### No Future Information

Model inputs must not include:

- Columns beginning with `future_`
- Columns beginning with `target_`
- Forward returns
- Future minimum or maximum prices
- Any information unavailable on the prediction date

### Same Comparison Dataset

Random Forest and deep learning models must be evaluated using:

- The same eligible rows
- The same features
- The same target definitions
- The same chronological splits
- The same missing-data rules
- The same evaluation metrics
- The same historical snapshot dates

---

## Initial Deep Learning Scope

The first deep learning model will be a tabular multilayer perceptron.

Its input will be one stock-date row containing the same 79 engineered features used by the Random Forest models.

```text
79 engineered features
        ↓
Dense hidden layers
        ↓
Normalization and activation
        ↓
Dropout regularization
        ↓
Binary prediction probability
```

Separate neural networks will initially be trained for:

- NIFTY 50 outperformance
- 10% downside risk

Sequence models such as LSTM, GRU, temporal CNN, and Transformer models will be considered only after the tabular benchmark is complete.

---

## Evaluation Metrics

### Classification Metrics

- ROC-AUC
- Precision-recall AUC
- Log loss
- Brier score
- Precision
- Recall
- F1 score
- Confusion matrix
- Probability calibration

### Opportunity Ranking Evaluation

- High Opportunity versus Low Opportunity future return
- NIFTY 50 excess-return difference
- Outperformance rate
- Monthly stability
- Bootstrap confidence intervals

### Downside-Risk Evaluation

- Q1 Highest Risk versus Q5 Lowest Risk
- 5% downside-event difference
- 10% downside-event difference
- Worst-path return difference
- Downside-event recall
- Bootstrap confidence intervals

---

## Model Promotion Criteria

A deep learning candidate will only replace or complement the Random Forest baseline when it:

1. Improves untouched out-of-time results.
2. Produces stable results across multiple periods.
3. Maintains acceptable probability calibration.
4. Improves historical ranking behavior.
5. Avoids all forms of temporal and target leakage.
6. Can be reproduced from saved configuration and model artifacts.
7. Can be used efficiently in the production snapshot pipeline.

Possible final outcomes are:

```text
Deep learning performs better
        → Promote the neural-network model

An ensemble performs better
        → Combine Random Forest and neural-network probabilities

Random Forest remains stronger
        → Retain the Random Forest baseline
```

Retaining the simpler model is a valid result when deep learning does not demonstrate reliable improvement.

---

## Expected Outputs

This notebook should produce:

- Dataset audit summary
- Model-feature list
- Target definitions
- Date-range summary
- Class-distribution tables
- Chronological split definition
- Split-level row and stock counts
- Leakage-audit results
- Random Forest benchmark configuration
- Deep learning experiment configuration
- Saved audit metadata for later notebooks

---

## Notebook Status

**Stage:** Deep learning preparation and data audit  
**Model training:** Not started  
**Production models:** Random Forest baseline retained  
**Next notebook:** Tabular MLP baseline

## Baseline Audit Findings

The existing Random Forest benchmark was recovered from
`06_baseline_modeling.ipynb`.

### Original Chronological Split

| Split | Date Rule | Rows |
|---|---|---:|
| Train | Date on or before 2023-12-31 | 253,538 |
| Validation | 2024-01-01 through 2024-12-31 | 21,780 |
| Test | Date on or after 2025-01-01 | 32,135 |

The same split and the same 79 ordered features were used for both prediction
targets.

### Targets

Outperformance target:

`target_outperform_nifty50_20d`

Downside-risk target:

`target_big_downside_10pct_20d`

### Target Balance

| Target | Train | Validation | Test |
|---|---:|---:|---:|
| Outperformance | 51.29% | 50.45% | 51.73% |
| 10% downside | 14.76% | 12.54% | 11.20% |

The outperformance target is approximately balanced.

The downside target is imbalanced and requires class-aware evaluation using
ROC-AUC, precision-recall AUC, recall, calibration, and ranking analysis.

### Feature Audit

- Final feature count: 79
- Future columns included: 0
- Target columns included: 0
- Readiness flags included: 0
- Infinite values found: 0
- Only one feature contains missing values:
  `close_position_in_day_range`
- Maximum missing rate: approximately 0.28%
- Existing pipelines use median imputation

### Random Forest Outperformance Benchmark

| Metric | Validation | Test |
|---|---:|---:|
| ROC-AUC | 0.5404 | 0.5179 |
| Average precision | 0.5399 | 0.5307 |

The outperformance model is a weak classification benchmark and is the primary
candidate for improvement.

### Random Forest Downside Benchmark

| Metric | Test |
|---|---:|
| ROC-AUC | 0.6894 |
| Average precision | 0.2107 |
| Precision | 19.76% |
| Recall | 53.61% |

The downside model is the stronger production baseline.

### Split-Boundary Concern

The original split does not include an embargo between train, validation, and
test periods.

Because both targets look forward 20 trading days, observations near a split
boundary may calculate their outcomes using dates belonging to the following
split.

This is target-window overlap rather than direct feature leakage.

### Deep Learning Evaluation Decision

Two benchmarks will be retained:

1. **Legacy benchmark**

   Uses the exact original split so deep learning can be compared directly with
   the saved Random Forest results.

2. **Purged benchmark**

   Removes the final 20 trading dates before validation and test boundaries so
   forward target windows cannot cross into the following split.

Model selection and production-promotion decisions will be based primarily on
the purged benchmark.

### Imports , dataset & Models loading

        Load the research dataset and both production models
        Recover the exact 79-feature order
        Recreate the original legacy split
        Find the final 20 observed trading dates before each split boundary
        Remove those dates from the earlier split
        Create reusable legacy_split_masks and purged_split_masks

In [1]:
from pathlib import Path

import joblib
import pandas as pd


# ---------------------------------------------------------
# Resolve project paths
# ---------------------------------------------------------

def find_project_root(start_path: Path) -> Path:
    """Find the repository root from the current notebook directory."""

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "config.yaml").exists() and (path / "README.md").exists():
            return path

    raise FileNotFoundError(
        "Could not find the project root containing config.yaml and README.md."
    )


def extract_feature_names(fitted_model) -> list[str]:
    """Extract the ordered feature names stored in a fitted sklearn pipeline."""

    if hasattr(fitted_model, "feature_names_in_"):
        return list(fitted_model.feature_names_in_)

    if hasattr(fitted_model, "named_steps"):
        for step in fitted_model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Could not find feature_names_in_ in the fitted model or pipeline."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

OUTPERFORM_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_downside_10pct_20d_v1.joblib"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "deep_learning_experiments"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Load dataset and production models
# ---------------------------------------------------------

data = pd.read_parquet(DATA_PATH)
data["date"] = pd.to_datetime(data["date"])

outperform_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_model = joblib.load(DOWNSIDE_MODEL_PATH)

outperform_features = extract_feature_names(outperform_model)
downside_features = extract_feature_names(downside_model)

if outperform_features != downside_features:
    raise ValueError(
        "The outperform and downside models do not use the same ordered features."
    )

feature_cols = outperform_features

if len(feature_cols) != 79:
    raise ValueError(
        f"Expected 79 model features, found {len(feature_cols)}."
    )


# ---------------------------------------------------------
# Recreate the original modeling population
# ---------------------------------------------------------

OUTPERFORM_TARGET = "target_outperform_nifty50_20d"
DOWNSIDE_TARGET = "target_big_downside_10pct_20d"
READY_COL = "target_ready_v1_20d"

required_columns = [
    "date",
    "yf_ticker",
    READY_COL,
    OUTPERFORM_TARGET,
    DOWNSIDE_TARGET,
    *feature_cols,
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise KeyError(
        "The research dataset is missing required columns:\n"
        + "\n".join(missing_columns)
    )

model_data = (
    data.loc[data[READY_COL].eq(1)]
    .sort_values(["date", "yf_ticker"])
    .reset_index(drop=True)
    .copy()
)


# ---------------------------------------------------------
# Define the original legacy split
# ---------------------------------------------------------

TRAIN_END_DATE = pd.Timestamp("2023-12-31")
VALID_START_DATE = pd.Timestamp("2024-01-01")
VALID_END_DATE = pd.Timestamp("2024-12-31")
TEST_START_DATE = pd.Timestamp("2025-01-01")

legacy_train_mask = model_data["date"].le(TRAIN_END_DATE)

legacy_valid_mask = (
    model_data["date"].ge(VALID_START_DATE)
    & model_data["date"].le(VALID_END_DATE)
)

legacy_test_mask = model_data["date"].ge(TEST_START_DATE)

legacy_split_masks = {
    "train": legacy_train_mask,
    "valid": legacy_valid_mask,
    "test": legacy_test_mask,
}


# ---------------------------------------------------------
# Calculate the 20-trading-day purge windows
# ---------------------------------------------------------

PURGE_TRADING_DAYS = 20

legacy_train_dates = pd.DatetimeIndex(
    model_data.loc[legacy_train_mask, "date"]
    .drop_duplicates()
    .sort_values()
)

legacy_valid_dates = pd.DatetimeIndex(
    model_data.loc[legacy_valid_mask, "date"]
    .drop_duplicates()
    .sort_values()
)

if len(legacy_train_dates) <= PURGE_TRADING_DAYS:
    raise ValueError(
        "The training period does not contain enough dates for the purge."
    )

if len(legacy_valid_dates) <= PURGE_TRADING_DAYS:
    raise ValueError(
        "The validation period does not contain enough dates for the purge."
    )

train_purge_dates = legacy_train_dates[-PURGE_TRADING_DAYS:]
valid_purge_dates = legacy_valid_dates[-PURGE_TRADING_DAYS:]


# Remove the final 20 trading dates from train and validation.
purged_train_mask = (
    legacy_train_mask
    & ~model_data["date"].isin(train_purge_dates)
)

purged_valid_mask = (
    legacy_valid_mask
    & ~model_data["date"].isin(valid_purge_dates)
)

# The test split has no later evaluation split, so its tail is not purged.
purged_test_mask = legacy_test_mask.copy()

purged_split_masks = {
    "train": purged_train_mask,
    "valid": purged_valid_mask,
    "test": purged_test_mask,
}


# ---------------------------------------------------------
# Create the split audit table
# ---------------------------------------------------------

audit_rows = []

for split_method, split_masks in [
    ("legacy", legacy_split_masks),
    ("purged", purged_split_masks),
]:
    for split_name, split_mask in split_masks.items():
        split_data = model_data.loc[split_mask]

        audit_rows.append(
            {
                "split_method": split_method,
                "split": split_name,
                "rows": len(split_data),
                "stocks": split_data["yf_ticker"].nunique(),
                "trading_dates": split_data["date"].nunique(),
                "start_date": split_data["date"].min(),
                "end_date": split_data["date"].max(),
                "outperform_positive_rate": split_data[
                    OUTPERFORM_TARGET
                ].mean(),
                "downside_positive_rate": split_data[
                    DOWNSIDE_TARGET
                ].mean(),
            }
        )

split_audit = pd.DataFrame(audit_rows)

split_audit["outperform_positive_pct"] = (
    split_audit["outperform_positive_rate"] * 100
)

split_audit["downside_positive_pct"] = (
    split_audit["downside_positive_rate"] * 100
)


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print("Project root:", PROJECT_ROOT)
print("Dataset shape:", data.shape)
print("Modeling population:", model_data.shape)
print("Unique stocks:", model_data["yf_ticker"].nunique())
print("Model features:", len(feature_cols))

print("\nTraining purge window:")
print(
    train_purge_dates.min().date(),
    "to",
    train_purge_dates.max().date(),
)
print("Trading dates removed:", len(train_purge_dates))

print("\nValidation purge window:")
print(
    valid_purge_dates.min().date(),
    "to",
    valid_purge_dates.max().date(),
)
print("Trading dates removed:", len(valid_purge_dates))

display(
    split_audit[
        [
            "split_method",
            "split",
            "rows",
            "stocks",
            "trading_dates",
            "start_date",
            "end_date",
            "outperform_positive_pct",
            "downside_positive_pct",
        ]
    ]
)

Project root: E:\Projects\marketguard-india
Dataset shape: (357370, 200)
Modeling population: (307453, 200)
Unique stocks: 90
Model features: 79

Training purge window:
2023-12-01 to 2023-12-29
Trading dates removed: 20

Validation purge window:
2024-12-03 to 2024-12-31
Trading dates removed: 20


,split_method,split,rows,stocks,trading_dates,start_date,end_date,outperform_positive_pct,downside_positive_pct
0,legacy,train,253538,90,3156,2011-01-03,2023-12-29,51.292508,14.759918
1,legacy,valid,21780,90,242,2024-01-02,2024-12-31,50.454545,12.543618
2,legacy,test,32135,90,359,2025-01-01,2026-06-16,51.728645,11.196515
3,purged,train,251738,90,3136,2011-01-03,2023-11-30,51.214755,14.851552
4,purged,valid,19980,90,222,2024-01-02,2024-12-02,51.126126,10.490490
5,purged,test,32135,90,359,2025-01-01,2026-06-16,51.728645,11.196515


## Chronological Split Audit Results

The original model population contains:

- 307,453 stock-date rows
- 90 stocks
- 79 ordered model features
- Dates from 2011 through 2026

### Legacy Split

| Split | Rows | Trading Dates | Start | End |
|---|---:|---:|---|---|
| Train | 253,538 | 3,156 | 2011-01-03 | 2023-12-29 |
| Validation | 21,780 | 242 | 2024-01-02 | 2024-12-31 |
| Test | 32,135 | 359 | 2025-01-01 | 2026-06-16 |

The legacy split exactly reproduces the split used in the original Random Forest
modeling notebook.

### Purge Windows

Because both targets look forward 20 trading days, the final 20 observed trading
dates were removed from the earlier split before each boundary.

Training purge window:

`2023-12-01` through `2023-12-29`

Validation purge window:

`2024-12-03` through `2024-12-31`

Each purge removed:

`20 trading dates × 90 stocks = 1,800 rows`

### Purged Split

| Split | Rows | Trading Dates | Start | End |
|---|---:|---:|---|---|
| Train | 251,738 | 3,136 | 2011-01-03 | 2023-11-30 |
| Validation | 19,980 | 222 | 2024-01-02 | 2024-12-02 |
| Test | 32,135 | 359 | 2025-01-01 | 2026-06-16 |

The test period remains unchanged because there is no later evaluation split
whose boundary must be protected.

### Target Rates

| Method | Split | Outperformance Rate | 10% Downside Rate |
|---|---|---:|---:|
| Legacy | Train | 51.29% | 14.76% |
| Legacy | Validation | 50.45% | 12.54% |
| Legacy | Test | 51.73% | 11.20% |
| Purged | Train | 51.21% | 14.85% |
| Purged | Validation | 51.13% | 10.49% |
| Purged | Test | 51.73% | 11.20% |

### Interpretation

The outperformance target remains approximately balanced across all periods.

The downside target is imbalanced and changes across market periods. In
particular, removing December 2024 reduces the validation downside-event rate
from 12.54% to 10.49%.

This demonstrates that downside frequency is sensitive to the selected market
period. Deep learning models must therefore be evaluated using:

1. Precision-recall metrics
2. Probability calibration
3. Cross-sectional risk ranking
4. Historical monthly stability
5. The untouched test period

The legacy split will be retained for direct comparison with the saved Random
Forest results.

The purged split will be the primary split for model selection and production
promotion decisions.

### Final data-contract , leakage audit and save reusable audit artifacts.

In [2]:
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Feature-contract audit
# ---------------------------------------------------------

duplicate_features = (
    pd.Series(feature_cols)
    .loc[pd.Series(feature_cols).duplicated()]
    .tolist()
)

leakage_prefixes = ("future_", "target_")

suspicious_feature_names = [
    feature
    for feature in feature_cols
    if feature.startswith(leakage_prefixes)
]

non_numeric_features = [
    feature
    for feature in feature_cols
    if not pd.api.types.is_numeric_dtype(model_data[feature])
]

missing_feature_rows = []

for feature in feature_cols:
    feature_values = pd.to_numeric(
        model_data[feature],
        errors="coerce",
    )

    missing_feature_rows.append(
        {
            "feature": feature,
            "missing_count": int(feature_values.isna().sum()),
            "missing_pct": float(feature_values.isna().mean() * 100),
            "infinite_count": int(
                np.isinf(feature_values.dropna().to_numpy()).sum()
            ),
        }
    )

feature_quality_audit = (
    pd.DataFrame(missing_feature_rows)
    .sort_values(
        ["missing_pct", "feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Target-completeness audit
# ---------------------------------------------------------

target_audit_rows = []

for split_method, split_masks in [
    ("legacy", legacy_split_masks),
    ("purged", purged_split_masks),
]:
    for split_name, split_mask in split_masks.items():
        split_frame = model_data.loc[split_mask]

        target_audit_rows.append(
            {
                "split_method": split_method,
                "split": split_name,
                "rows": len(split_frame),
                "outperform_missing_count": int(
                    split_frame[OUTPERFORM_TARGET].isna().sum()
                ),
                "downside_missing_count": int(
                    split_frame[DOWNSIDE_TARGET].isna().sum()
                ),
                "outperform_class_0_count": int(
                    split_frame[OUTPERFORM_TARGET].eq(0).sum()
                ),
                "outperform_class_1_count": int(
                    split_frame[OUTPERFORM_TARGET].eq(1).sum()
                ),
                "downside_class_0_count": int(
                    split_frame[DOWNSIDE_TARGET].eq(0).sum()
                ),
                "downside_class_1_count": int(
                    split_frame[DOWNSIDE_TARGET].eq(1).sum()
                ),
            }
        )

target_audit = pd.DataFrame(target_audit_rows)


# ---------------------------------------------------------
# Split-integrity audit
# ---------------------------------------------------------

def count_mask_overlap(
    first_mask: pd.Series,
    second_mask: pd.Series,
) -> int:
    """Count rows assigned to both supplied masks."""

    return int((first_mask & second_mask).sum())


split_integrity = {
    "legacy_train_valid_overlap": count_mask_overlap(
        legacy_train_mask,
        legacy_valid_mask,
    ),
    "legacy_train_test_overlap": count_mask_overlap(
        legacy_train_mask,
        legacy_test_mask,
    ),
    "legacy_valid_test_overlap": count_mask_overlap(
        legacy_valid_mask,
        legacy_test_mask,
    ),
    "purged_train_valid_overlap": count_mask_overlap(
        purged_train_mask,
        purged_valid_mask,
    ),
    "purged_train_test_overlap": count_mask_overlap(
        purged_train_mask,
        purged_test_mask,
    ),
    "purged_valid_test_overlap": count_mask_overlap(
        purged_valid_mask,
        purged_test_mask,
    ),
}

legacy_assigned_mask = (
    legacy_train_mask
    | legacy_valid_mask
    | legacy_test_mask
)

purged_assigned_mask = (
    purged_train_mask
    | purged_valid_mask
    | purged_test_mask
)

legacy_unassigned_rows = int((~legacy_assigned_mask).sum())
purged_unassigned_rows = int((~purged_assigned_mask).sum())

expected_purged_rows = (
    len(train_purge_dates) + len(valid_purge_dates)
) * model_data["yf_ticker"].nunique()


# ---------------------------------------------------------
# Hard validation checks
# ---------------------------------------------------------

if duplicate_features:
    raise ValueError(
        f"Duplicate model features found: {duplicate_features}"
    )

if suspicious_feature_names:
    raise ValueError(
        "Future or target columns were found in the model features: "
        f"{suspicious_feature_names}"
    )

if non_numeric_features:
    raise TypeError(
        f"Non-numeric model features found: {non_numeric_features}"
    )

if feature_quality_audit["infinite_count"].sum() != 0:
    raise ValueError(
        "Infinite values were found in the model features."
    )

if (
    target_audit[
        [
            "outperform_missing_count",
            "downside_missing_count",
        ]
    ]
    .to_numpy()
    .sum()
    != 0
):
    raise ValueError(
        "Missing target values were found in an evaluation split."
    )

if any(value != 0 for value in split_integrity.values()):
    raise ValueError(
        f"Overlapping split masks found: {split_integrity}"
    )

if legacy_unassigned_rows != 0:
    raise ValueError(
        f"Legacy split left {legacy_unassigned_rows} rows unassigned."
    )

if purged_unassigned_rows != expected_purged_rows:
    raise ValueError(
        "Unexpected number of purged rows. "
        f"Expected {expected_purged_rows}, "
        f"found {purged_unassigned_rows}."
    )


# ---------------------------------------------------------
# Create purge-date audit
# ---------------------------------------------------------

purge_date_audit = pd.concat(
    [
        pd.DataFrame(
            {
                "boundary": "train_to_validation",
                "purged_from_split": "train",
                "date": train_purge_dates,
            }
        ),
        pd.DataFrame(
            {
                "boundary": "validation_to_test",
                "purged_from_split": "valid",
                "date": valid_purge_dates,
            }
        ),
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# Save audit artifacts
# ---------------------------------------------------------

feature_list_path = (
    REPORT_DIR
    / "deep_learning_feature_list_v1.csv"
)

feature_quality_path = (
    REPORT_DIR
    / "deep_learning_feature_quality_audit_v1.csv"
)

split_audit_path = (
    REPORT_DIR
    / "deep_learning_split_audit_v1.csv"
)

target_audit_path = (
    REPORT_DIR
    / "deep_learning_target_audit_v1.csv"
)

purge_dates_path = (
    REPORT_DIR
    / "deep_learning_purge_dates_v1.csv"
)

metadata_path = (
    REPORT_DIR
    / "deep_learning_data_contract_v1.json"
)

pd.DataFrame(
    {
        "feature_order": range(1, len(feature_cols) + 1),
        "feature": feature_cols,
    }
).to_csv(
    feature_list_path,
    index=False,
)

feature_quality_audit.to_csv(
    feature_quality_path,
    index=False,
)

split_audit.to_csv(
    split_audit_path,
    index=False,
)

target_audit.to_csv(
    target_audit_path,
    index=False,
)

purge_date_audit.to_csv(
    purge_dates_path,
    index=False,
)

data_contract = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(DATA_PATH),
    "outperform_model_path": str(OUTPERFORM_MODEL_PATH),
    "downside_model_path": str(DOWNSIDE_MODEL_PATH),
    "modeling_rows": int(len(model_data)),
    "stock_count": int(model_data["yf_ticker"].nunique()),
    "feature_count": int(len(feature_cols)),
    "targets": {
        "outperform": OUTPERFORM_TARGET,
        "downside": DOWNSIDE_TARGET,
    },
    "legacy_split": {
        "train_end_date": str(TRAIN_END_DATE.date()),
        "validation_start_date": str(VALID_START_DATE.date()),
        "validation_end_date": str(VALID_END_DATE.date()),
        "test_start_date": str(TEST_START_DATE.date()),
    },
    "purge": {
        "trading_days": PURGE_TRADING_DAYS,
        "train_purge_start": str(train_purge_dates.min().date()),
        "train_purge_end": str(train_purge_dates.max().date()),
        "validation_purge_start": str(valid_purge_dates.min().date()),
        "validation_purge_end": str(valid_purge_dates.max().date()),
        "purged_rows": purged_unassigned_rows,
    },
    "audit": {
        "duplicate_feature_count": len(duplicate_features),
        "suspicious_feature_count": len(suspicious_feature_names),
        "non_numeric_feature_count": len(non_numeric_features),
        "features_with_missing_values": int(
            feature_quality_audit["missing_count"].gt(0).sum()
        ),
        "total_infinite_values": int(
            feature_quality_audit["infinite_count"].sum()
        ),
        "legacy_unassigned_rows": legacy_unassigned_rows,
        "purged_unassigned_rows": purged_unassigned_rows,
        "split_overlaps": split_integrity,
    },
}

metadata_path.write_text(
    json.dumps(
        data_contract,
        indent=2,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------
# Display final audit results
# ---------------------------------------------------------

print("Feature count:", len(feature_cols))
print("Duplicate features:", len(duplicate_features))
print("Future/target features:", len(suspicious_feature_names))
print("Non-numeric features:", len(non_numeric_features))
print(
    "Features with missing values:",
    feature_quality_audit["missing_count"].gt(0).sum(),
)
print(
    "Total infinite values:",
    feature_quality_audit["infinite_count"].sum(),
)
print("Legacy unassigned rows:", legacy_unassigned_rows)
print("Purged unassigned rows:", purged_unassigned_rows)
print("Expected purged rows:", expected_purged_rows)
print("Split overlaps:", split_integrity)

print("\nFeature quality rows requiring attention:")
display(
    feature_quality_audit.loc[
        feature_quality_audit["missing_count"].gt(0)
        | feature_quality_audit["infinite_count"].gt(0)
    ]
)

print("\nTarget audit:")
display(target_audit)

print("\nSaved audit artifacts:")

for saved_path in [
    feature_list_path,
    feature_quality_path,
    split_audit_path,
    target_audit_path,
    purge_dates_path,
    metadata_path,
]:
    print(saved_path)

Feature count: 79
Duplicate features: 0
Future/target features: 0
Non-numeric features: 0
Features with missing values: 1
Total infinite values: 0
Legacy unassigned rows: 0
Purged unassigned rows: 3600
Expected purged rows: 3600
Split overlaps: {'legacy_train_valid_overlap': 0, 'legacy_train_test_overlap': 0, 'legacy_valid_test_overlap': 0, 'purged_train_valid_overlap': 0, 'purged_train_test_overlap': 0, 'purged_valid_test_overlap': 0}

Feature quality rows requiring attention:


,feature,missing_count,missing_pct,infinite_count
0,close_position_in_day_range,281,0.091396,0



Target audit:


,split_method,split,rows,outperform_missing_count,downside_missing_count,outperform_class_0_count,outperform_class_1_count,downside_class_0_count,downside_class_1_count
0,legacy,train,253538,0,0,123492,130046,216116,37422
1,legacy,valid,21780,0,0,10791,10989,19048,2732
2,legacy,test,32135,0,0,15512,16623,28537,3598
3,purged,train,251738,0,0,122811,128927,214351,37387
4,purged,valid,19980,0,0,9765,10215,17884,2096
5,purged,test,32135,0,0,15512,16623,28537,3598



Saved audit artifacts:
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_feature_list_v1.csv
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_feature_quality_audit_v1.csv
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_split_audit_v1.csv
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_target_audit_v1.csv
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_purge_dates_v1.csv
E:\Projects\marketguard-india\reports\deep_learning_experiments\deep_learning_data_contract_v1.json


## Final Data-Contract Audit

The deep learning research dataset passed all required validation checks.

### Feature Contract

| Check | Result |
|---|---:|
| Ordered model features | 79 |
| Duplicate features | 0 |
| `future_*` or `target_*` features | 0 |
| Non-numeric features | 0 |
| Features containing infinite values | 0 |
| Features containing missing values | 1 |

The only feature containing missing observations is:

`close_position_in_day_range`

It contains:

- 281 missing rows
- 0.0914% of the modeling population
- No infinite values

This is a very small missing-data rate. The neural-network preprocessing
pipeline will use median imputation fitted exclusively on the training split.

### Target Contract

Both targets are complete in every legacy and purged split:

- `target_outperform_nifty50_20d`
- `target_big_downside_10pct_20d`

No target values are missing.

### Split Integrity

All pairwise split overlaps are zero for both the legacy and purged methods.

| Check | Result |
|---|---:|
| Legacy unassigned rows | 0 |
| Purged unassigned rows | 3,600 |
| Expected purged rows | 3,600 |
| Train-validation overlap | 0 |
| Train-test overlap | 0 |
| Validation-test overlap | 0 |

The 3,600 unassigned rows correspond exactly to:

- 1,800 rows removed before the validation boundary
- 1,800 rows removed before the test boundary

Each purge contains 20 trading dates across 90 stocks.

### Final Modeling Population

| Split Method | Split | Rows | Outperformance Class 1 | Downside Class 1 |
|---|---|---:|---:|---:|
| Legacy | Train | 253,538 | 130,046 | 37,422 |
| Legacy | Validation | 21,780 | 10,989 | 2,732 |
| Legacy | Test | 32,135 | 16,623 | 3,598 |
| Purged | Train | 251,738 | 128,927 | 37,387 |
| Purged | Validation | 19,980 | 10,215 | 2,096 |
| Purged | Test | 32,135 | 16,623 | 3,598 |

### Preprocessing Rule for Deep Learning

All preprocessing must be learned from the training split only.

The intended process is:

```text
Training features
        ↓
Fit median imputer
        ↓
Fit feature scaler
        ↓
Transform training, validation, and test data
```

The validation and test periods must never be used to calculate:

- Median imputation values
- Feature means
- Feature standard deviations
- Class weights
- Decision thresholds
- Early-stopping parameters
- Model-selection criteria

### Audit Artifacts

The following reusable artifacts were generated:

- `deep_learning_feature_list_v1.csv`
- `deep_learning_feature_quality_audit_v1.csv`
- `deep_learning_split_audit_v1.csv`
- `deep_learning_target_audit_v1.csv`
- `deep_learning_purge_dates_v1.csv`
- `deep_learning_data_contract_v1.json`

### Data-Audit Decision

The dataset is approved for deep learning experiments.

The purged chronological split will be the primary model-selection benchmark.

The legacy split will be retained only for direct comparison with the original
Random Forest results.

The next research stage is the tabular MLP baseline.